# 🔮 RateIQ – App Store Rating Prediction
### Complete ML Pipeline: EDA → Feature Engineering → Model Training → Evaluation
---

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

plt.style.use('dark_background')
sns.set_palette('husl')
np.random.seed(42)
print('✅ Imports complete')

## 2. Data Loading & Cleaning

In [ ]:
# Load data (generated by models/train_model.py)
df = pd.read_csv('../data/apps.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nDtypes:\n{df.dtypes}')
df.head()

In [ ]:
# Missing values & duplicates
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'\nRating range: {df.rating.min()} – {df.rating.max()}')
df.describe()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('RateIQ – EDA Overview', fontsize=18, fontweight='bold', y=1.01)

# 1. Rating distribution
axes[0,0].hist(df['rating'], bins=30, color='#6366F1', edgecolor='none', alpha=.85)
axes[0,0].set_title('Rating Distribution')
axes[0,0].set_xlabel('Rating'); axes[0,0].set_ylabel('Count')

# 2. Category avg rating
cat_avg = df.groupby('category')['rating'].mean().sort_values(ascending=True).tail(15)
cat_avg.plot(kind='barh', ax=axes[0,1], color='#22C55E', alpha=.85)
axes[0,1].set_title('Avg Rating by Category (Top 15)')

# 3. Installs vs Rating
sample = df.sample(500)
axes[0,2].scatter(np.log1p(sample['installs']), sample['rating'],
                  alpha=.35, color='#FBBF24', s=20)
axes[0,2].set_title('log(Installs) vs Rating')
axes[0,2].set_xlabel('log(Installs)'); axes[0,2].set_ylabel('Rating')

# 4. Free vs Paid
df.groupby(df['price'].apply(lambda x: 'Paid' if x > 0 else 'Free'))['rating'].mean().plot(
    kind='bar', ax=axes[1,0], color=['#6366F1','#F87171'], alpha=.85, rot=0)
axes[1,0].set_title('Avg Rating: Free vs. Paid')

# 5. Ads impact
df.groupby('has_ads')['rating'].mean().rename({0:'No Ads',1:'Has Ads'}).plot(
    kind='bar', ax=axes[1,1], color=['#22C55E','#F87171'], alpha=.85, rot=0)
axes[1,1].set_title('Avg Rating: Ads Impact')

# 6. Correlation heatmap
num_cols = ['rating','size_mb','price','reviews','update_days','num_screenshots','has_ads','is_free']
corr = df[num_cols].corr()
sns.heatmap(corr, ax=axes[1,2], annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=.5)
axes[1,2].set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.savefig('../data/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Engineering

In [ ]:
le_cat = LabelEncoder(); le_cr = LabelEncoder()
df['category_enc'] = le_cat.fit_transform(df['category'])
df['content_rating_enc'] = le_cr.fit_transform(df['content_rating'])
df['log_installs'] = np.log1p(df['installs'])
df['log_reviews'] = np.log1p(df['reviews'])

FEATURES = ['category_enc','size_mb','log_installs','price',
            'content_rating_enc','log_reviews','update_days',
            'num_screenshots','has_ads','is_free']

X = df[FEATURES]; y = df['rating']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f'Train: {X_train_s.shape}, Test: {X_test_s.shape}')

## 5. Model Training & Comparison

In [ ]:
models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest':    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                    learning_rate=0.05, random_state=42),
}

results = {}
for name, m in models.items():
    m.fit(X_train_s, y_train)
    y_pred = m.predict(X_test_s)
    results[name] = {
        'MAE':  mean_absolute_error(y_test, y_pred),
        'RMSE': mean_squared_error(y_test, y_pred)**0.5,
        'R2':   r2_score(y_test, y_pred),
    }
    print(f'{name:25s}  MAE={results[name]["MAE"]:.4f}  RMSE={results[name]["RMSE"]:.4f}  R²={results[name]["R2"]:.4f}')

results_df = pd.DataFrame(results).T
results_df

In [ ]:
# Visual comparison
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#6366F1','#22C55E','#FBBF24']
for i, metric in enumerate(['MAE','RMSE','R2']):
    vals = results_df[metric]
    bars = ax[i].bar(vals.index, vals.values, color=colors, alpha=.85)
    ax[i].set_title(metric, fontweight='bold')
    ax[i].set_xticklabels(vals.index, rotation=20, ha='right', fontsize=9)
    for bar, val in zip(bars, vals.values):
        ax[i].text(bar.get_x()+bar.get_width()/2, bar.get_height()+.002,
                   f'{val:.4f}', ha='center', fontsize=9)
plt.suptitle('Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Best Model – Feature Importance

In [ ]:
best_model = models['Gradient Boosting']
importances = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='barh', ax=ax, color='#6366F1', alpha=.85)
ax.set_title('Gradient Boosting – Feature Importance', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout(); plt.show()

## 7. Save Final Artifacts

In [ ]:
artifacts = {
    'model': best_model, 'scaler': scaler,
    'le_cat': le_cat, 'le_cr': le_cr,
    'features': FEATURES,
    'metrics': results['Gradient Boosting'],
    'categories': list(le_cat.classes_),
    'content_ratings': list(le_cr.classes_),
}
with open('../models/model_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)
print('✅ Model artifacts saved to models/model_artifacts.pkl')